# 04 — Star Schema Build

Creates dimension-ready customer/date data and documents the fact-table grain.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, year, month, quarter, dayofweek
spark=SparkSession.builder.appName('RetailStarSchema').getOrCreate()
base='../Datasets/'
customers=spark.read.option('header',True).option('inferSchema',True).csv(base+'customers.csv')
orders=spark.read.option('header',True).option('inferSchema',True).csv(base+'orders_new.csv')


In [ ]:
dim_customer=(customers.dropDuplicates(['customer_id'])
    .select('customer_id','customer_fname','customer_lname','city','state','pincode'))

fact_orders=(orders.dropDuplicates(['order_id'])
    .select('order_id','order_date','customer_id','order_status'))


In [ ]:
dim_date=(fact_orders.select(to_date('order_date').alias('calendar_date')).distinct()
    .withColumn('date_key', year('calendar_date')*10000+month('calendar_date')*100+dayofweek('calendar_date'))
    .withColumn('calendar_year', year('calendar_date'))
    .withColumn('month_number', month('calendar_date'))
    .withColumn('quarter_number', quarter('calendar_date')))

print('dim_customer rows:', dim_customer.count())
print('fact_orders rows:', fact_orders.count())
print('dim_date rows:', dim_date.count())
